# PF + Known-Portion Calibration Hybrid

Particle Filter (GR-based TVT tracking) + Likelihood-based seed weighting.

Pipeline:
1. Run PF with 64 seeds → get 64 TVT predictions
2. Weight each seed by GR-tracking likelihood (softmax temperature)
3. Force known rows to TVT_input
4. Ensemble weighted average → submission

In [ ]:
import os
from glob import glob
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

# ---- Paths ----
DATA_DIR = '/kaggle/input/competitions/rogii-wellbore-geology-prediction'
TEST_DIR = os.path.join(DATA_DIR, 'test')
SAMPLE_SUB_PATH = os.path.join(DATA_DIR, 'sample_submission.csv')
OUTPUT_PATH = '/kaggle/working/submission.csv'

# ---- PF Configuration ----
N_PARTICLES = 500
N_SEEDS = 64
LIK_TEMPERATURE = 3.0

# ---- PF noise parameters ----
MOM = 0.998
VN = 0.002
PN = 0.005
RP = 0.1
RR = 0.001
RESAMP_THRESH = 0.5

In [ ]:
def run_particle_filter(hw, tw, n_particles=N_PARTICLES, seed=42,
                        mom=MOM, vn=VN, pn=PN, rp=RP, rr=RR):
    """Conservative Particle Filter for TVT tracking via GR matching."""
    tw_s = tw.sort_values('TVT')
    tw_tvt = tw_s['TVT'].values.astype(float)
    tw_gr = tw_s['GR'].fillna(tw_s['GR'].mean()).values.astype(float)

    kn = hw[hw['TVT_input'].notna()]
    ev = hw[hw['TVT_input'].isna()]
    if len(ev) == 0:
        return hw['TVT_input'].values.astype(float).copy(), 0.0

    last = kn.iloc[-1]
    last_tvt = float(last['TVT_input'])
    last_Z = float(last['Z'])
    last_MD = float(last['MD'])

    tw_at_k = np.interp(kn['TVT_input'].values, tw_tvt, tw_gr)
    gs = float(np.clip(np.nanstd(kn['GR'].fillna(0).values - tw_at_k), 10., 60.))

    tail = kn.tail(30)
    dt = np.diff(tail['TVT_input'].values)
    dz = np.diff(tail['Z'].values)
    dm = np.diff(tail['MD'].values)
    m = dm > 0
    ir = float(np.median((dt + dz)[m] / dm[m])) if m.sum() >= 3 else 0.0

    N = n_particles
    rng = np.random.default_rng(seed)
    ls = last_tvt + last_Z
    pos = ls + 2.0 * rng.standard_normal(N)
    rate = ir + 0.01 * rng.standard_normal(N)
    w = np.ones(N) / N

    md_v = ev['MD'].values.astype(float)
    z_v = ev['Z'].values.astype(float)
    gr_interp = hw['GR'].interpolate(limit_direction='both').fillna(tw_gr.mean())
    gr_v = gr_interp.values.astype(float)[ev.index]

    out_vals = hw['TVT_input'].values.astype(float).copy()
    res = np.empty(len(ev))
    prev_MD = last_MD
    log_lik = 0.0

    for i in range(len(ev)):
        dm_step = max(md_v[i] - prev_MD, 1.0)
        rate = mom * rate + vn * rng.standard_normal(N)
        pos = pos + rate * dm_step + pn * rng.standard_normal(N)
        tvt_p = pos - z_v[i]
        tvt_p = np.clip(tvt_p, tw_tvt[0] - 100, tw_tvt[-1] + 100)
        pos = tvt_p + z_v[i]

        eg = np.interp(tvt_p, tw_tvt, tw_gr)
        d = (gr_v[i] - eg) / gs
        lk = np.exp(-0.5 * np.minimum(d**2, 600.))
        lk = np.maximum(lk, 1e-300)
        avg_lk = float((w * lk).sum())
        log_lik += np.log(max(avg_lk, 1e-300))
        w = w * lk
        ws = w.sum()
        w = w / ws if ws > 0 else np.ones(N) / N

        n_eff = 1.0 / (w**2).sum()
        if n_eff < RESAMP_THRESH * N:
            cum = np.cumsum(w)
            u0 = rng.uniform(0, 1.0 / N)
            idx = np.clip(np.searchsorted(cum, u0 + np.arange(N) / N), 0, N - 1)
            pos = pos[idx] + rp * rng.standard_normal(N)
            rate = rate[idx] + rr * rng.standard_normal(N)
            w = np.ones(N) / N

        res[i] = float(np.dot(w, pos - z_v[i]))
        prev_MD = md_v[i]

    out_vals[list(ev.index)] = res
    return out_vals, log_lik


def predict_well_pf_calibrated(well_id, hw, tw, **kwargs):
    """PF ensemble with likelihood-based seed weighting."""
    known_mask = hw['TVT_input'].notna().values
    predict_mask = ~known_mask

    if predict_mask.sum() < 1:
        return hw['TVT_input'].values.astype(float).copy()

    preds = []
    liks = []
    for s in range(N_SEEDS):
        try:
            p, ll = run_particle_filter(hw, tw, n_particles=N_PARTICLES, seed=s)
            preds.append(p)
            liks.append(ll)
        except Exception:
            fallback = hw['TVT_input'].fillna(
                hw['TVT_input'].mean() if known_mask.any() else 0.0
            ).values.astype(float).copy()
            preds.append(fallback)
            liks.append(-1e9)

    if len(preds) == 0:
        return hw['TVT_input'].fillna(0.0).values.astype(float).copy()

    pred_arr = np.stack(preds, 0)
    liks = np.array(liks)

    # Likelihood-based softmax weighting
    liks_n = liks - liks.max()
    weights = np.exp(liks_n / LIK_TEMPERATURE)
    weights /= weights.sum()

    # Weighted PF ensemble
    tvt_pred = (weights[:, None] * pred_arr).sum(axis=0)

    # Force known anchors to exact values
    if known_mask.any():
        tvt_pred[known_mask] = hw.loc[known_mask, 'TVT_input'].values.astype(float)

    tvt_pred = np.nan_to_num(tvt_pred, nan=0.0, posinf=0.0, neginf=0.0)
    tw_tvt = tw['TVT'].values.astype(float)
    tvt_pred = np.clip(tvt_pred, tw_tvt.min() - 200, tw_tvt.max() + 200)

    return tvt_pred


def generate_submission(test_dir, sample_sub_path, output_path, predict_func, **kwargs):
    sub = pd.read_csv(sample_sub_path)
    test_files = sorted(glob(os.path.join(test_dir, '*__horizontal_well.csv')))
    predictions = {}

    for f in test_files:
        well_id = os.path.basename(f).replace('__horizontal_well.csv', '')
        hw = pd.read_csv(f)
        tw = pd.read_csv(os.path.join(test_dir, f'{well_id}__typewell.csv'))
        tvt_pred = np.asarray(predict_func(well_id, hw, tw, **kwargs), dtype=float)
        predictions[well_id] = np.nan_to_num(tvt_pred, nan=0.0, posinf=0.0, neginf=0.0)
        print(f'  Predicted {well_id}: {len(hw)} rows, '
              f'TVT range [{predictions[well_id].min():.1f}, {predictions[well_id].max():.1f}]')

    for idx, row in sub.iterrows():
        well_id = row['id'].rsplit('_', 1)[0]
        row_idx = int(row['id'].rsplit('_', 1)[1])
        if well_id in predictions:
            sub.at[idx, 'tvt'] = predictions[well_id][row_idx]

    sub.to_csv(output_path, index=False)
    print(f'\nSubmission saved to {output_path}')
    return sub

In [ ]:
print('=' * 60)
print('PF + Known-Portion Calibration Hybrid')
print('=' * 60)
print(f'Particles: {N_PARTICLES} | Seeds: {N_SEEDS} | LIK_TEMP: {LIK_TEMPERATURE}')
print(f'Noise: MOM={MOM}, VN={VN}, PN={PN}, RP={RP}, RR={RR}')

print('\nGenerating submission...')
submission = generate_submission(
    TEST_DIR,
    SAMPLE_SUB_PATH,
    OUTPUT_PATH,
    predict_well_pf_calibrated,
)

print(f'\nTotal rows: {len(submission)}, TVT range: '
      f'[{submission["tvt"].min():.1f}, {submission["tvt"].max():.1f}]')
submission.head()